In [5]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


In [3]:

# Example DataFrame structure (replace with your data)
path = "merged_traffic_data.csv"
df = pd.read_csv(path, delimiter=';', encoding='utf-8', )
df.head()

,Land,Strklas,Strnum,Wotag,Fahrtzw,Stunde,KFZ_R1,KFZ_R2,Monat,plz,lat,lon,station_id,inhab_plz,density,inhab_100
0,10,A,6,6,s,1,128,77,1,90610.0,49.397955,11.305518,101,4074.0,163.16819,2423864.0
1,10,A,6,6,s,2,220,164,1,90610.0,49.397955,11.305518,101,4074.0,163.16819,2423864.0
2,10,A,6,6,s,3,204,137,1,90610.0,49.397955,11.305518,101,4074.0,163.16819,2423864.0
3,10,A,6,6,s,4,124,104,1,90610.0,49.397955,11.305518,101,4074.0,163.16819,2423864.0
4,10,A,6,6,s,5,79,78,1,90610.0,49.397955,11.305518,101,4074.0,163.16819,2423864.0


In [10]:
# convert str klas to int
df['Strklas'].unique()

array(['A'], dtype=object)

In [6]:
# Simulate some sample data for demonstration
# Features and target
X = df[['Land', 'Strnum', 'Wotag', 'Stunde', 'Monat','inhab_plz', 'density', 'inhab_100']].values
y = df['KFZ_R1'].values

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


# Build an upgraded model
model = Sequential([
    Dense(128, activation='relu', input_shape=X.shape[1:]),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer
])

# Compile model
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# train with early stopping
model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    verbose=1
)

# Save the model
model.save('traffic_volume_model_v2.h5')

Epoch 1/100


/home/thore/anaconda3/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


 10248/190092 ━━━━━━━━━━━━━━━━━━━━ 3:23 1ms/step - loss: 1150492.7500 - mae: 823.4822

KeyboardInterrupt: 

In [ ]:
# test model 

# Load the model
loaded_model = tf.keras.models.load_model('traffic_volume_model.h5')
# Predict traffic volume for a specific time

# Example input data for prediction (replace with actual values)
# ['Land', 'Strnum', 'Wotag', 'Stunde', 'KFZ_R1', 'KFZ_R2', 'Monat','inhab_plz', 'density', 'inhab_100']
input_data = np.array([[1, 8, 2, 8, 5, 12345, 0.5, 500000]])  # Example input

# predict
input_data_scaled = scaler.transform(input_data)
predicted_volume = loaded_model.predict(input_data_scaled)

print(f"Predicted traffic volume for Tuesday at 8AM: {predicted_volume[0][0]:.2f}")
